# MotoGP Events Held - Advanced Modeling and Analysis

**Dataset Focus**: `grand_prix_events_held.csv`  
**CRISP-DM Phase**: 4 - Modeling  
**Purpose**: Advanced analysis of event hosting patterns, circuit importance, and geographical distribution

## Business Focus
- Circuit hosting frequency and importance modeling
- Geographical expansion patterns
- Event sustainability analysis

In [ ]:
# Setup and data loading
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Load prepared events data
data_path = Path("../../data/interim/")
df = pd.read_csv(data_path / "events_held_prepared.csv")

print(f"Events held data loaded: {df.shape}")
print(f"Date range: {df['season'].min()} - {df['season'].max()}")
print(f"Unique circuits: {df['track_clean'].nunique()}")
print(f"Unique countries: {df['country_clean'].nunique()}")
df.head()

## Circuit Importance Modeling

In [ ]:
print("=== CIRCUIT IMPORTANCE MODELING ===")

# Calculate circuit metrics
circuit_metrics = df.groupby('track_clean').agg({
    'season': ['count', 'min', 'max', 'nunique'],
    'country_clean': 'first'
})

circuit_metrics.columns = ['total_events', 'first_event', 'last_event', 'seasons_active', 'country']
circuit_metrics['hosting_span'] = circuit_metrics['last_event'] - circuit_metrics['first_event']
circuit_metrics['events_per_season'] = circuit_metrics['total_events'] / circuit_metrics['seasons_active']
circuit_metrics['longevity_score'] = circuit_metrics['hosting_span'] * circuit_metrics['events_per_season']

# Circuit importance categories
def categorize_circuit_importance(row):
    if row['total_events'] >= 50:
        return 'Historic Venue'
    elif row['total_events'] >= 20:
        return 'Established Circuit'
    elif row['total_events'] >= 10:
        return 'Regular Host'
    elif row['total_events'] >= 5:
        return 'Occasional Host'
    else:
        return 'Limited Hosting'

circuit_metrics['importance_category'] = circuit_metrics.apply(categorize_circuit_importance, axis=1)

print("Circuit importance distribution:")
importance_dist = circuit_metrics['importance_category'].value_counts()
for category, count in importance_dist.items():
    percentage = count / len(circuit_metrics) * 100
    avg_events = circuit_metrics[circuit_metrics['importance_category'] == category]['total_events'].mean()
    print(f"{category}: {count} circuits ({percentage:.1f}%) - Avg: {avg_events:.1f} events")

# Top circuits analysis
print(f"\nTop 15 circuits by total events:")
top_circuits = circuit_metrics.nlargest(15, 'total_events')
for i, (circuit, data) in enumerate(top_circuits.iterrows(), 1):
    events = int(data['total_events'])
    span = int(data['hosting_span'])
    country = data['country']
    rate = data['events_per_season']
    print(f"{i:2d}. {circuit} ({country}): {events} events over {span} years ({rate:.1f}/season)")

# Visualization
plt.figure(figsize=(14, 10))
top_15 = circuit_metrics.nlargest(15, 'total_events')
plt.barh(range(len(top_15)), top_15['total_events'], color='lightblue')
plt.yticks(range(len(top_15)), [name[:25] + '...' if len(name) > 25 else name for name in top_15.index])
plt.xlabel('Total Events Hosted')
plt.title('Top 15 Circuits by Event Hosting Frequency')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Geographical Expansion Modeling

In [ ]:
print("=== GEOGRAPHICAL EXPANSION MODELING ===")

# Country-level analysis
country_metrics = df.groupby('country_clean').agg({
    'track_clean': 'nunique',
    'season': ['count', 'min', 'max', 'nunique']
})

country_metrics.columns = ['unique_circuits', 'total_events', 'first_event', 'last_event', 'seasons_active']
country_metrics['events_per_circuit'] = country_metrics['total_events'] / country_metrics['unique_circuits']
country_metrics['hosting_span'] = country_metrics['last_event'] - country_metrics['first_event']

# Continental analysis
continental_analysis = df.groupby('continent').agg({
    'country_clean': 'nunique',
    'track_clean': 'nunique',
    'season': 'count'
})
continental_analysis.columns = ['countries', 'circuits', 'total_events']
continental_analysis = continental_analysis.sort_values('total_events', ascending=False)

print("Continental distribution:")
for continent, data in continental_analysis.iterrows():
    countries = int(data['countries'])
    circuits = int(data['circuits'])
    events = int(data['total_events'])
    print(f"{continent}: {events} events ({countries} countries, {circuits} circuits)")

# Top hosting countries
print(f"\nTop 15 hosting countries:")
top_countries = country_metrics.nlargest(15, 'total_events')
for i, (country, data) in enumerate(top_countries.iterrows(), 1):
    events = int(data['total_events'])
    circuits = int(data['unique_circuits'])
    events_per_circuit = data['events_per_circuit']
    span = int(data['hosting_span'])
    print(f"{i:2d}. {country}: {events} events ({circuits} circuits, {events_per_circuit:.1f} avg/circuit, {span}yr span)")

# Expansion timeline analysis
print(f"\n=== EXPANSION TIMELINE ANALYSIS ===")
decade_expansion = df.groupby(['decade', 'continent']).agg({
    'country_clean': 'nunique',
    'track_clean': 'nunique'
}).reset_index()

print("New countries hosting by decade and continent:")
for decade in sorted(df['decade'].unique()):
    decade_data = decade_expansion[decade_expansion['decade'] == decade]
    total_countries = decade_data['country_clean'].sum()
    total_circuits = decade_data['track_clean'].sum()
    print(f"{decade}s: {total_countries} countries, {total_circuits} circuits active")
    for _, row in decade_data.iterrows():
        print(f"  {row['continent']}: {row['country_clean']} countries")

# Visualization: Geographic distribution
plt.figure(figsize=(12, 8))
continental_analysis['total_events'].plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Distribution of MotoGP Events by Continent')
plt.ylabel('')  # Remove ylabel
plt.show()

# Hosting concentration analysis
print(f"\n=== HOSTING CONCENTRATION ANALYSIS ===")
total_events = len(df)
top_5_countries_events = country_metrics.nlargest(5, 'total_events')['total_events'].sum()
concentration_5 = top_5_countries_events / total_events * 100

top_10_countries_events = country_metrics.nlargest(10, 'total_events')['total_events'].sum()
concentration_10 = top_10_countries_events / total_events * 100

print(f"Top 5 countries host {concentration_5:.1f}% of all events")
print(f"Top 10 countries host {concentration_10:.1f}% of all events")

# Calculate Herfindahl-Hirschman Index for concentration
country_shares = country_metrics['total_events'] / total_events
hhi = sum(share**2 for share in country_shares)
print(f"HHI concentration index: {hhi:.3f} (closer to 0 = more distributed)")

## Circuit Clustering and Sustainability

In [ ]:
print("=== CIRCUIT CLUSTERING AND SUSTAINABILITY ===")

# Prepare features for clustering
if len(circuit_metrics) >= 10:
    clustering_features = ['total_events', 'hosting_span', 'events_per_season', 'longevity_score']
    X = circuit_metrics[clustering_features].fillna(0)
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # K-means clustering
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    circuit_metrics['cluster'] = clusters
    
    # Analyze clusters
    cluster_names = {
        0: 'Intermittent Hosts',
        1: 'Established Venues',
        2: 'Historic Circuits',
        3: 'Modern Additions'
    }
    
    print("Circuit clusters:")
    for cluster_id in range(4):
        cluster_circuits = circuit_metrics[circuit_metrics['cluster'] == cluster_id]
        if len(cluster_circuits) > 0:
            avg_events = cluster_circuits['total_events'].mean()
            avg_span = cluster_circuits['hosting_span'].mean()
            print(f"\nCluster {cluster_id} ({len(cluster_circuits)} circuits):")
            print(f"  Average events: {avg_events:.1f}")
            print(f"  Average span: {avg_span:.1f} years")
            
            # Show representative circuits
            top_circuits = cluster_circuits.nlargest(3, 'total_events')
            print(f"  Examples: {list(top_circuits.index)[:3]}")

# Sustainability analysis
print(f"\n=== SUSTAINABILITY ANALYSIS ===")

# Currently active circuits (hosted events in last decade)
recent_decade = df['season'].max() - 10
active_circuits = df[df['season'] >= recent_decade]['track_clean'].unique()
inactive_circuits = set(circuit_metrics.index) - set(active_circuits)

print(f"Circuit activity status:")
print(f"  Active (last 10 years): {len(active_circuits)} circuits")
print(f"  Inactive (>10 years): {len(inactive_circuits)} circuits")
print(f"  Retention rate: {len(active_circuits)/(len(active_circuits)+len(inactive_circuits))*100:.1f}%")

if len(inactive_circuits) > 0:
    print(f"\nTop inactive circuits by historical importance:")
    inactive_metrics = circuit_metrics.loc[list(inactive_circuits)].nlargest(5, 'total_events')
    for circuit, data in inactive_metrics.iterrows():
        events = int(data['total_events'])
        last_event = int(data['last_event'])
        years_inactive = df['season'].max() - last_event
        print(f"  {circuit}: {events} events, last held in {last_event} ({years_inactive} years ago)")

# Future hosting potential
print(f"\n=== HOSTING POTENTIAL ANALYSIS ===")
recent_additions = circuit_metrics[circuit_metrics['first_event'] >= 2000].nlargest(10, 'total_events')

if len(recent_additions) > 0:
    print(f"Most successful recent additions (since 2000):")
    for i, (circuit, data) in enumerate(recent_additions.iterrows(), 1):
        events = int(data['total_events'])
        first = int(data['first_event'])
        rate = data['events_per_season']
        country = data['country']
        print(f"  {i}. {circuit} ({country}): {events} events since {first} ({rate:.1f}/year)")

print(f"\n✅ EVENTS HELD MODELING COMPLETED")
print(f"Key insights: Clear geographical concentration with European dominance,")
print(f"historic venues showing strong sustainability, and ongoing global expansion.")